In [1]:
import re
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# 机器学习核心库
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# 采样与文本处理 (BERT)
from imblearn.pipeline import Pipeline as ImbPipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# UI 美化
from rich.console import Console
from rich.table import Table
from rich import box

# 初始化
tqdm.pandas()
console = Console()
device = "cuda" if torch.cuda.is_available() else "cpu"


# 1. 加载【分词器】 (Tokenizer)
tokenizer = AutoTokenizer.from_pretrained(
    "jackietung/bert-base-chinese-sentiment-finetuned"
)
# 2. 加载【序列分类模型】 (Model)
model = AutoModelForSequenceClassification.from_pretrained(
    "jackietung/bert-base-chinese-sentiment-finetuned"
)
model.eval()  # 推理模式
def get_sentiment_scores_batch(
    texts,
    batch_size=64,
    device="cpu"  
):
    """
    批量计算中文文本情感得分
    返回：np.ndarray，长度等于 texts
    """
    model.to(device)
    model.eval()

    scores = []

    texts = texts.fillna("").astype(str).tolist()

    for i in tqdm(range(0, len(texts), batch_size), desc="BERT 批量情感推理"):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)

        # 假设 label=1 是正向
        batch_scores = probs[:, 1].cpu().numpy()
        scores.extend(batch_scores)

    return np.array(scores)

def load_rents(is_train=True):
    if is_train:
        paths = [
            "./data/raw_data/ruc_Class25Q2_train_rent.csv",
            "/mnt/c/A_Class_Git/B_class_private/Exams/Exam_Class25Q2/ruc_Class25Q2_train_rent.csv",
        ]
        for p in paths:
            if Path(p).exists():
                return pd.read_csv(p, low_memory=False)
        raise FileNotFoundError("❌ 未找到 rent 数据文件，请检查路径")
    else:
        paths = [
            "./data/raw_data/ruc_Class25Q2_test_rent.csv",
            "/mnt/c/A_Class_Git/B_class_private/Exams/Exam_Class25Q2/ruc_Class25Q2_test_rent.csv",
        ]
        for p in paths:
            if Path(p).exists():
                return pd.read_csv(p, low_memory=False)
        raise FileNotFoundError("❌ 未找到 rent 测试数据文件，请检查路径")

def load_sales(is_train=True):
    if is_train:
        paths = [
            "./data/raw_data/ruc_Class25Q2_train_price.csv",
            "/mnt/c/A_Class_Git/B_class_private/Exams/Exam_Class25Q2/ruc_Class25Q2_train_price.csv",
        ]
        for p in paths:
            if Path(p).exists():
                return pd.read_csv(p, low_memory=False)
        raise FileNotFoundError("❌ 未找到 sales 数据文件，请检查路径")
    else:
        paths = [
            "./data/raw_data/ruc_Class25Q2_test_price.csv",
            "/mnt/c/A_Class_Git/B_class_private/Exams/Exam_Class25Q2/ruc_Class25Q2_test_price.csv",
        ]
        for p in paths:
            if Path(p).exists():
                return pd.read_csv(p, low_memory=False)
        raise FileNotFoundError("❌ 未找到 sales 测试数据文件，请检查路径")

def remove_outliers_iqr(df, target_col='Price', factor=1.5):
    """
    使用 IQR 剔除目标列的异常值
    """
    Q1 = df[target_col].quantile(0.25)
    Q3 = df[target_col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - factor * IQR
    upper_bound = Q3 + factor * IQR
    
    # 过滤数据
    before = len(df)
    df_clean = df[(df[target_col] >= lower_bound) & (df[target_col] <= upper_bound)].copy()
    after = len(df_clean)
    
    print(f"IQR 清理 [{target_col}]: 移除 {before - after} 行, 剩余 {after} 行")
    return df_clean

def preprocess_rent_data(df, is_train=True):
    df = df.copy()

    if is_train:
        before_rows = len(df)

        # 计算每一行的缺失比例
        row_missing_ratio = df.isna().mean(axis=1)

        # 缺失率阈值：30%
        threshold = 0.50

        # 保留缺失率 < 30% 的样本
        df = df[row_missing_ratio < threshold].copy()

        if 'Price' in df.columns:
                    df = remove_outliers_iqr(df, target_col='Price', factor=3.0)

        after_rows = len(df)
        removed_rows = before_rows - after_rows

        print(f"🧹 数据清洗完成：总计移除 {before_rows - after_rows} 行样本")

    # --- 1. 数值提取类 (从带有单位或混乱格式的字符串中提取 float) ---
    def clean_num(x):
        if pd.isna(x) or x == '暂无': return np.nan
        res = re.findall(r'(\d+\.?\d*)', str(x))
        return float(res[0]) if res else np.nan

    # 面积、绿化率、物业费、燃气费、供热费、房屋总数、楼栋总数、停车费用
    num_cols = ['面积', '绿 化 率', '物 业 费', '燃气费', '供热费', '房屋总数', '楼栋总数', '停车费用']
    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_num)
            # 填充缺失值为中位数
            df[col] = df[col].fillna(df[col].median() if is_train else 0)

    # --- 2. 户型拆解 (室、厅、卫) ---
    def split_layout(x):
        room = re.findall(r'(\d+)(?:室|房间)', str(x))
        hall = re.findall(r'(\d+)厅', str(x))
        bath = re.findall(r'(\d+)卫', str(x))
        return pd.Series([
            int(room[0]) if room else 1, # 默认起码1室
            int(hall[0]) if hall else 0,
            int(bath[0]) if bath else 1
        ])
    df[['室', '厅', '卫']] = df['户型'].apply(split_layout)

    # --- 3. 朝向处理 (多标签简化为 4 个主方向) ---
    for d in ['东', '南', '西', '北']:
        df[f'朝向_{d}'] = df['朝向'].astype(str).apply(lambda x: 1 if d in x else 0)

    # --- 4. 楼层映射 (高中低) ---
    def get_floor_level(x):
        x = str(x)
        if '高' in x: return 2
        if '中' in x: return 1
        if '低' in x: return 0
        # 处理数字格式 如 19/46层
        match = re.match(r'(\d+)/(\d+)', x)
        if match:
            ratio = int(match.group(1)) / int(match.group(2))
            return 2 if ratio > 0.66 else (0 if ratio < 0.33 else 1)
        return 1 # 默认中楼层
    df['楼层等级'] = df['楼层'].apply(get_floor_level)

    # --- 5. 二值化处理 (有/无, 民/商) ---
    # 装修 (精装=1, None=0)
    df['精装修'] = df['装修'].apply(lambda x: 1 if x == '精装修' else 0)
    # 电梯、燃气 (有=1, 无=0)
    for col in ['电梯', '燃气']:
        df[col] = df[col].map({'有': 1, '无': 0}).fillna(0)
    # 用水、用电 (民=1, 商=0)
    for col in ['用水', '用电']:
        df[col] = df[col].astype(str).apply(lambda x: 1 if '民' in x else 0)
    # 租赁方式 (整租=1, 合租=0)
    df['租赁方式'] = df['租赁方式'].map({'整租': 1, '合租': 0}).fillna(1)

    # --- 6. 时间特征 ---
    df['交易时间'] = pd.to_datetime(df['交易时间'], errors='coerce')
    df['交易月份'] = df['交易时间'].dt.month.fillna(6) # 默认6月
    # 建筑年代处理：提取第一个4位数字作为年份
    df['建成年份'] = df['建筑年代'].astype(str).str.extract(r'(\d{4})').astype(float)
    df['建成年份'] = df['建成年份'].fillna(df['建成年份'].median() if is_train else 2010)
    df['房龄'] = 2024 - df['建成年份']

    # --- 7. 配套设施 (计数特征) ---
    # 统计有多少种家电，这通常直接影响租金
    df['配套数量'] = df['配套设施'].astype(str).apply(lambda x: 0 if x == 'nan' else len(x.split('、')))

    # if '客户反馈' in df.columns:
    #     df.drop(columns=['客户反馈'], inplace=True)
    # --- 7.x 客户反馈：BERT 情感特征 ---
    if '客户反馈' in df.columns:
        print("使用 BERT 提取客户反馈情感特征...")

        sentiment_scores = get_sentiment_scores_batch(
            df['客户反馈'],
            batch_size=128,      # CPU 推荐 16–32，GPU 可 64+
            device="cpu"        # 有 GPU 改成 "cuda"
        )

        df['客户反馈'] = sentiment_scores

        # 缺失值处理（中位数）
        df['客户反馈'] = df['客户反馈'].fillna(
            df['客户反馈'].median() if is_train else 0.5
        )

    # --- 8. 删除无法直接入模的原始长文本和无效列 ---
    drop_list = [
        '户型', '装修', '楼层', '朝向', '交易时间', '付款方式', '建筑年代', 
        '配套设施', '产权描述', '物业公司', '开发商', '物业类别', 
        '物业办公电话', '建成年份','车位',"租期","供水","供暖","供电","建筑结构",'环线位置'
    ]
    df.drop(columns=[c for c in drop_list if c in df.columns], inplace=True)

    # --- 9. 处理剩下的分类变量 (One-Hot) ---
    # 针对 环线位置、采暖、建筑结构 等进行编码
    cat_cols = ['环线位置', '采暖', '建筑结构', '用水', '用电']
    df = pd.get_dummies(df, columns=[c for c in cat_cols if c in df.columns], dummy_na=False)

    # --- 10. 数值列缺失值统一用中位数填充 ---
    num_cols = df.select_dtypes(include=[np.number]).columns

    for col in num_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median() if is_train else 0)
    
    df.columns = df.columns.str.replace(r"\s+", "_", regex=True)

    return df

def preprocess_sale_data(df, is_train=True):
    df = df.copy()
    
    if is_train:
        before_rows = len(df)

        # 计算每一行的缺失比例
        row_missing_ratio = df.isna().mean(axis=1)

        # 缺失率阈值：30%
        threshold = 0.50

        # 保留缺失率 < 30% 的样本
        df = df[row_missing_ratio < threshold].copy()

        if 'Price' in df.columns:
                    df = remove_outliers_iqr(df, target_col='Price', factor=3.0)

        after_rows = len(df)
        removed_rows = before_rows - after_rows

        print(f"数据清洗完成：总计移除 {before_rows - after_rows} 行样本")

    # --- 1. 基础数值清洗 (建筑面积, 物业费, 绿化率等) ---
    def extract_float(x):
        if pd.isna(x): return np.nan
        res = re.findall(r'(\d+\.?\d*)', str(x))
        return float(res[0]) if res else np.nan

    num_cols = ['建筑面积', '套内面积', '绿 化 率', '物 业 费', '房屋总数', '楼栋总数', '停车费用', '容 积 率','燃气费','供热费']
    for col in num_cols:
        if col in df.columns:
            df[col] = df[col].apply(extract_float)
            # 训练集用中位数填充，测试集保持一致
            if is_train:
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(0)

    # --- 2. 房屋户型拆解 (更细致：室/厅/厨/卫) ---
    if '房屋户型' in df.columns:
        def split_detailed_layout(x):
            room = re.findall(r'(\d+)室', str(x))
            hall = re.findall(r'(\d+)厅', str(x))
            kitchen = re.findall(r'(\d+)厨', str(x))
            bath = re.findall(r'(\d+)卫', str(x))
            return pd.Series([
                int(room[0]) if room else 0,
                int(hall[0]) if hall else 0,
                int(kitchen[0]) if kitchen else 0,
                int(bath[0]) if bath else 0
            ])
        df[['室', '厅', '厨', '卫']] = df['房屋户型'].apply(split_detailed_layout)

    # --- 3. 楼层与总层数 (所在楼层) ---
    # 格式示例: "高楼层 (共6层)" -> 提取 楼层位置 和 总层数
    if '所在楼层' in df.columns:
        df['总层数'] = df['所在楼层'].str.extract(r'共(\d+)层').astype(float)
        df['楼层等级'] = df['所在楼层'].apply(lambda x: 2 if '高' in str(x) else (0 if '低' in str(x) else 1))
        df['总层数'] = df['总层数'].fillna(df['总层数'].median() if is_train else 10)

    # --- 4. 梯户比例 (计算密集度) ---
    # 格式示例: "一梯两户" -> 0.5
    if '梯户比例' in df.columns:
        def calc_lift_ratio(x):
            mapping = {'一': 1, '两': 2, '二': 2, '三': 3, '四': 4, '五': 5, '六': 6, '七': 7, '八': 8, '九': 9, '十': 10}
            nums = re.findall(r'([一两二三四五六七八九十\d])梯([一两二三四五六七八九十\d])户', str(x))
            if nums:
                t, h = nums[0]
                t_val = mapping.get(t, t) if not str(t).isdigit() else t
                h_val = mapping.get(h, h) if not str(h).isdigit() else h
                try:
                    return float(t_val) / float(h_val)
                except:
                    return 0.5
            return 0.5
        df['梯户比数值'] = df['梯户比例'].apply(calc_lift_ratio)

    # --- 5. 朝向、电梯、权属等二值化 ---
    # 朝向
    for d in ['东', '南', '西', '北']:
        df[f'朝向_{d}'] = df['房屋朝向'].astype(str).apply(lambda x: 1 if d in x else 0)
    
    # 电梯
    if '配备电梯' in df.columns:
        df['有电梯'] = df['配备电梯'].map({'有': 1, '无': 0}).fillna(0)
        
    # 满五/满二 (房屋年限)
    if '房屋年限' in df.columns:
        df['满五'] = df['房屋年限'].apply(lambda x: 1 if '满五年' in str(x) else 0)
        df['满二'] = df['房屋年限'].apply(lambda x: 1 if '满' in str(x) else 0)

    # 产权所属
    if '产权所属' in df.columns:
        df['共有产权'] = df['产权所属'].map({'共有': 1, '非共有': 0}).fillna(0)

    # --- 6. 装修情况 (映射为等级) ---
    if '装修情况' in df.columns:
        # 精装=3, 简装=2, 其他=1, 毛坯=0
        dec_map = {'精装': 3, '简装': 2, '其他': 1, '毛坯': 0}
        df['装修等级'] = df['装修情况'].map(dec_map).fillna(1)
        df.drop(columns=['装修情况'], inplace=True)

    # --- 7. 时间特征 (上次交易到现在的年数) ---
    current_year = 2025
    if '上次交易' in df.columns:
        df['持有年限'] = pd.to_datetime(df['上次交易'], errors='coerce').dt.year
        df['持有年限'] = current_year - df['持有年限']
        df['持有年限'] = df['持有年限'].fillna(df['持有年限'].median() if is_train else 5)

    # if '客户反馈' in df.columns:
    #     df.drop(columns=['客户反馈'], inplace=True)
    # --- 7.x 客户反馈：BERT 情感特征 ---
    if '客户反馈' in df.columns:
        print("使用 BERT 提取客户反馈情感特征...")

        sentiment_scores = get_sentiment_scores_batch(
            df['客户反馈'],
            batch_size=128,    
            device="cpu"     
        )

        df['客户反馈'] = sentiment_scores

        # 缺失值处理（中位数）
        df['客户反馈'] = df['客户反馈'].fillna(
            df['客户反馈'].median() if is_train else 0.5
        )

    # --- 8. 删除文本和低信息量列 ---
    # 别墅类型、抵押信息(全空)、核心卖点(长文本)等
    cols_to_drop = [
        '房屋户型', '所在楼层', '房屋朝向', '梯户比例', '配备电梯', '交易时间', 
        '交易权属', '上次交易', '房屋用途', '房屋年限', '产权所属', '抵押信息',
        '房屋优势', '核心卖点', '户型介绍', '周边配套', '交通出行', '建筑结构',
        '环线', '环线位置', '物业类别', '建筑年代', '开发商', '物业公司',
        '物业办公电话', '产权描述', '供水', '供暖', '供电', '建筑结构_comm','别墅类型'
    ]
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

    # --- 9. 处理剩下的分类变量 ---
    # 针对 建筑结构 这种只有 7 个值的字段进行 One-Hot
    # 之前删除了，如果想保留可以移出 drop 列表

    # --- 10. 数值列缺失值统一用中位数填充 ---
    num_cols = df.select_dtypes(include=[np.number]).columns

    for col in num_cols:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median() if is_train else 0)
    
    df.columns = df.columns.str.replace(r"\s+", "_", regex=True)
    return df


In [2]:
def train_xgboost(X_train, y_train, random_state=42):
    """
    针对房产价格/租金预测的 XGBoost 回归训练
    """
    model = XGBRegressor(
        objective="reg:squarederror", # 回归任务
        n_estimators=2000,
        learning_rate=0.02,
        max_depth=7,                  # 房产特征多，稍微加深深度
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=random_state,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    return model


def train_logistic(X_train, y_train, random_state=42):
    """
    线性回归：标准化，并使用 Ridge 防止过拟合
    """
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=1.0, random_state=random_state))
    ])

    model.fit(X_train, y_train)
    return model


def train_random_forest(X_train, y_train, random_state=42):
    """
    随机森林回归
    """
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=15,             # 限制深度防止模型过大
        min_samples_leaf=5,       # 增加泛化能力
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    return model

from lightgbm import LGBMRegressor

def train_lightgbm(X_train, y_train, random_state=42):
    """
    LightGBM 回归模型（非线性，GBDT）
    """
    model = LGBMRegressor(
        objective="regression",
        n_estimators=800,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=random_state,
        force_row_wise=True,
        n_jobs=-1,
        verbose=-1
    )

    model.fit(X_train, y_train)
    return model

def evaluate_models_cv(model_builders, X, y, n_splits=5, random_state=42):
    """
    对多个回归模型进行交叉验证。
    """
    results = {}
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for name, builder in tqdm(model_builders.items(), desc="Models", leave=True):
        print(f"\n正在评估模型: {name}...")

        # 用于存储每一折的指标
        in_rmse_list, out_rmse_list = [], []
        mae_list, r2_list = [], []

        for fold, (train_idx, val_idx) in enumerate(
            tqdm(kf.split(X, y), total=n_splits, desc=f"{name} | CV", leave=False)
        ):
            # 1. 划分数据
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

            # 2. 目标变量对数化
            y_train_log = np.log1p(y_train_fold)

            # 3. 训练模型
            model = builder(X_train_fold, y_train_log, random_state=random_state)

            # --- 4. 样本内预测 (In-sample) ---
            y_in_pred_log = model.predict(X_train_fold)
            y_in_pred = np.expm1(y_in_pred_log)
            y_in_pred = np.maximum(y_in_pred, 0)
            in_rmse = np.sqrt(mean_squared_error(y_train_fold, y_in_pred))

            # --- 5. 样本外预测 (Out-of-sample) ---
            y_out_pred_log = model.predict(X_val_fold)
            y_out_pred = np.expm1(y_out_pred_log)
            y_out_pred = np.maximum(y_out_pred, 0)
            out_rmse = np.sqrt(mean_squared_error(y_val_fold, y_out_pred))

            # 其他指标 (基于样本外)
            mae = mean_absolute_error(y_val_fold, y_out_pred)
            r2 = r2_score(y_val_fold, y_out_pred)

            # 保存结果
            in_rmse_list.append(in_rmse)
            out_rmse_list.append(out_rmse)
            mae_list.append(mae)
            r2_list.append(r2)

            tqdm.write(
                f"   Fold {fold+1}: In-RMSE={in_rmse:.2f}, Out-RMSE={out_rmse:.2f}, R2={r2:.4f}"
            )

        # 汇总均值
        results[name] = {
            "In-sample RMSE": np.mean(in_rmse_list),
            "Out-of-sample RMSE": np.mean(out_rmse_list),
            "CV RMSE (Mean)": np.mean(out_rmse_list), # 交叉验证RMSE即样本外均值
            "MAE_mean": np.mean(mae_list),
            "R2_mean": np.mean(r2_list)
        }

    # ===== Rich 表格输出 (适配 PPT Metrics 表) =====
    results_df = pd.DataFrame(results).T
    console = Console()
    table = Table(
        title="🏆 期末考试模型性能指标表 (Metrics Table)",
        box=box.DOUBLE_EDGE,
        header_style="bold magenta"
    )

    table.add_column("Model", justify="left", style="bold cyan")
    table.add_column("In-sample RMSE", justify="right")
    table.add_column("Out-of-sample RMSE", justify="right")
    table.add_column("CV RMSE", justify="right")
    table.add_column("MAE (mean)", justify="right")
    table.add_column("R² (mean)", justify="right")

    for model_name, row in results_df.iterrows():
        table.add_row(
            model_name,
            f"{row['In-sample RMSE']:.2f}",
            f"{row['Out-of-sample RMSE']:.2f}",
            f"{row['CV RMSE (Mean)']:.2f}",
            f"{row['MAE_mean']:.2f}",
            f"{row['R2_mean']:.4f}"
        )

    console.print(table)
    return results_df

In [3]:
target = 'Price'
id_col = "ID"
model_builders = {
    "Logistic Regression": train_logistic,
    "Random Forest": train_random_forest,
    "XGBoost": train_xgboost,
    "LightGBM": train_lightgbm
}

# 处理租金数据
rent_df = preprocess_rent_data(load_rents(), is_train=True)

X_rent = rent_df.drop(columns=[target])
y_rent = rent_df[target]
feature_rent_cols = X_rent.columns.tolist()

# 处理售卖数据
sales_df = preprocess_sale_data(load_sales(), is_train=True)

X_sales = sales_df.drop(columns=[target])
y_sales = sales_df[target]
feature_sales_cols = X_sales.columns.tolist()


IQR 清理 [Price]: 移除 1935 行, 剩余 86912 行
🧹 数据清洗完成：总计移除 11987 行样本
使用 BERT 提取客户反馈情感特征...


BERT 批量情感推理:   0%|          | 0/679 [00:00<?, ?it/s]

IQR 清理 [Price]: 移除 2956 行, 剩余 93291 行
数据清洗完成：总计移除 10580 行样本
使用 BERT 提取客户反馈情感特征...


BERT 批量情感推理:   0%|          | 0/729 [00:00<?, ?it/s]

In [4]:

# --- 1. 租赁数据预测流程 ---
print("🚀 开始租赁数据处理与全量训练...")

# 评估并获取最佳模型名称
rent_results = evaluate_models_cv(
    model_builders,
    X_rent,
    y_rent,
    n_splits=5
)
rent_best_model_name = rent_results['MAE_mean'].idxmin()
print(f"✅ 租赁最佳模型确定为: {rent_best_model_name}")

# 获取对应的模型构建函数
best_rent_builder = model_builders[rent_best_model_name]

# 在对数空间进行全量训练
y_rent_log = np.log1p(y_rent)
rent_best_model = best_rent_builder(X_rent, y_rent_log, random_state=42)

# 加载并预处理测试集
rent_df_test = preprocess_rent_data(load_rents(is_train=False), is_train=False)

# 按照训练集的列顺序进行对齐
X_rent_test = rent_df_test.reindex(columns=X_rent.columns, fill_value=0)

# 执行预测并还原尺度
rent_pred_log = rent_best_model.predict(X_rent_test)
rent_pred = np.expm1(rent_pred_log)

# 防御性处理：确保预测值不小于训练集的最小值，防止极端异常值
rent_pred = np.maximum(rent_pred, y_rent.min())

rent_result = pd.DataFrame({
    id_col: rent_df_test[id_col].values,
    "Price": rent_pred
})

print("-" * 30)

# --- 2. 售卖数据预测流程 ---
print("🚀 开始售卖数据处理与全量训练...")

sale_results = evaluate_models_cv(
    model_builders,
    X_sales,
    y_sales,
    n_splits=5
)
sales_best_model_name = sale_results['MAE_mean'].idxmin()
print(f"✅ 售卖最佳模型确定为: {sales_best_model_name}")

best_sales_builder = model_builders[sales_best_model_name]

# 在对数空间进行全量训练
y_sales_log = np.log1p(y_sales)
sales_best_model = best_sales_builder(X_sales, y_sales_log, random_state=42)

# 加载并预处理测试集
sales_df_test = preprocess_sale_data(load_sales(is_train=False), is_train=False)

# 按照训练集的列顺序进行对齐
X_sales_test = sales_df_test.reindex(columns=X_sales.columns, fill_value=0)

# 执行预测并还原尺度
sales_pred_log = sales_best_model.predict(X_sales_test)
sales_pred = np.expm1(sales_pred_log)

# 防御性处理
sales_pred = np.maximum(sales_pred, y_sales.min())

sales_result = pd.DataFrame({
    id_col: sales_df_test[id_col].values,
    "Price": sales_pred
})

# --- 3. 结果合并与保存 ---
print("\n📊 正在合并结果并生成最终文件...")
final_result = pd.concat(
    [rent_result, sales_result],
    axis=0,
    ignore_index=True
)

# 确保 ID 排序正确，符合提交要求
final_result = final_result.sort_values(by=id_col)

# 保存文件
final_result.to_csv("./result.csv", index=False)
print("任务完成！结果已保存至 ./result.csv")


🚀 开始租赁数据处理与全量训练...


Models:   0%|          | 0/4 [00:00<?, ?it/s]


正在评估模型: Logistic Regression...


Logistic Regression | CV:   0%|          | 0/5 [00:00<?, ?it/s]

   Fold 1: In-RMSE=296987.23, Out-RMSE=309480.37, R2=0.4228
   Fold 2: In-RMSE=299553.95, Out-RMSE=294479.17, R2=0.4463
   Fold 3: In-RMSE=298669.98, Out-RMSE=295933.76, R2=0.4480
   Fold 4: In-RMSE=298206.71, Out-RMSE=296381.79, R2=0.4337
   Fold 5: In-RMSE=298813.01, Out-RMSE=296848.87, R2=0.4406

正在评估模型: Random Forest...


...

   Fold 1: In-RMSE=312013.35, Out-RMSE=342450.77, R2=0.9517
   Fold 2: In-RMSE=309337.96, Out-RMSE=350245.03, R2=0.9497
   Fold 3: In-RMSE=310938.10, Out-RMSE=354483.75, R2=0.9492
   Fold 4: In-RMSE=310171.37, Out-RMSE=351162.80, R2=0.9494
   Fold 5: In-RMSE=310412.23, Out-RMSE=348546.93, R2=0.9503


                             🏆 期末考试模型性能指标表 (Metrics Table)                             
╔═════════════════════╤════════════════╤════════════════════╤════════════╤════════════╤═══════════╗
║ Model               │ In-sample RMSE │ Out-of-sample RMSE │    CV RMSE │ MAE (mean) │ R² (mean) ║
╟─────────────────────┼────────────────┼────────────────────┼────────────┼────────────┼───────────╢
║ Logistic Regression │     1349855.32 │         1353307.28 │ 1353307.28 │  800490.84 │    0.2507 ║
║ Random Forest       │      299480.34 │          370886.59 │  370886.59 │  209151.44 │    0.9437 ║
║ XGBoost             │      231875.71 │          307138.72 │  307138.72 │  177211.50 │    0.9614 ║
║ LightGBM            │      310574.60 │          349377.86 │  349377.86 │  203219.64 │    0.9501 ║
╚═════════════════════╧════════════════╧════════════════════╧════════════╧════════════╧═══════════╝

✅ 售卖最佳模型确定为: XGBoost
使用 BERT 提取客户反馈情感特征...


BERT 批量情感推理:   0%|          | 0/266 [00:00<?, ?it/s]


📊 正在合并结果并生成最终文件...
任务完成！结果已保存至 ./result.csv


In [5]:
def get_oof_predictions(model_builder, X, y, n_splits=5, random_state=42):
    """
    生成 OOF (Out-of-fold) 预测结果，作为第二层模型的特征
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof_train = np.zeros(len(X))
    
    # 对数转换 y
    y_log = np.log1p(y)
    
    for train_idx, val_idx in kf.split(X, y_log):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr = y_log.iloc[train_idx]
        
        # 训练模型
        model = model_builder(X_tr, y_tr, random_state=random_state)
        
        # 预测验证集
        oof_train[val_idx] = model.predict(X_val)
        
    return oof_train

def train_stacking_model(X_train, y_train, X_test, model_builders):
    """
    执行 Stacking 融合
    """
    # 1. 准备第一层的特征 (OOF 预测)
    first_level_features = pd.DataFrame()
    test_level_features = pd.DataFrame()
    
    y_log = np.log1p(y_train)

    for name, builder in model_builders.items():
        if name == "Logistic Regression": continue # 线性模型通常不作为第一层基模型
        
        print(f"生成 {name} 的 OOF 特征...")
        # 训练集的 OOF 预测
        first_level_features[name] = get_oof_predictions(builder, X_train, y_train)
        
        # 测试集的全量预测 (均值化)
        full_model = builder(X_train, y_log, random_state=42)
        test_level_features[name] = full_model.predict(X_test)

    # 2. 第二层模型：使用简单的 Ridge 回归作为“裁判”
    # 裁判模型不需要太复杂，因为输入特征已经很少且很强了
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(first_level_features, y_log)
    
    # 3. 最终预测
    final_pred_log = meta_model.predict(test_level_features)
    final_pred = np.expm1(final_pred_log)
    
    return np.maximum(final_pred, y_train.min())

# 定义你想要参与融合的“专家”
stacking_builders = {
    "XGBoost": train_xgboost,
    "LightGBM": train_lightgbm,
    "Random Forest": train_random_forest
}

# 租赁数据的 Stacking
print("开始租赁数据 Stacking 融合...")
rent_pred_stack = train_stacking_model(X_rent, y_rent, X_rent_test, stacking_builders)

# 售卖数据的 Stacking
print("开始售卖数据 Stacking 融合...")
sales_pred_stack = train_stacking_model(X_sales, y_sales, X_sales_test, stacking_builders)

# --- 2. 包装成 DataFrame (关键步骤) ---
# 必须使用测试集的 ID 列进行对齐
rent_stack_df = pd.DataFrame({
    id_col: rent_df_test[id_col].values,
    "Price": rent_pred_stack
})

sales_stack_df = pd.DataFrame({
    id_col: sales_df_test[id_col].values,
    "Price": sales_pred_stack
})


# --- 3. 结果合并与保存 ---
print("\n📊 正在合并 Stacking 结果并生成最终文件...")
final_result = pd.concat(
    [rent_stack_df, sales_stack_df],
    axis=0,
    ignore_index=True
)

# 确保 ID 排序正确，符合提交要求
final_result = final_result.sort_values(by=id_col)

# 保存文件
final_result.to_csv("./result-stack.csv", index=False)
print("任务完成！Stacking 融合结果已保存至 ./result-stack.csv")

开始租赁数据 Stacking 融合...
生成 XGBoost 的 OOF 特征...
生成 LightGBM 的 OOF 特征...
生成 Random Forest 的 OOF 特征...
开始售卖数据 Stacking 融合...
生成 XGBoost 的 OOF 特征...
生成 LightGBM 的 OOF 特征...
生成 Random Forest 的 OOF 特征...

📊 正在合并 Stacking 结果并生成最终文件...
任务完成！Stacking 融合结果已保存至 ./result-stack.csv
